# Análise Exploratória de Dados (EDA) - Alfabetização Infantil
Neste notebook, realizaremos a EDA para compreender as variáveis que impactam a alfabetização.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Leitura e Estrutura dos Dados

In [ ]:
# Lendo o parquet
df = pd.read_parquet('../data/raw/abt_alunos_alfabetizacao.parquet')

# Exibindo as primeiras linhas e informações gerais
display(df.head())
df.info()

## 2. Análise da Variável Target
A variável alvo é `y_alfabetizado` (1 para alfabetizado, 0 para não alfabetizado).

In [ ]:
# Contagem de classes
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='y_alfabetizado', palette='Set2')
plt.title('Distribuição da Variável Target (y_alfabetizado)')
plt.xlabel('Alfabetizado (0 = Não, 1 = Sim)')
plt.ylabel('Quantidade de Alunos')

# Adicionando rótulos
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.show()

print("Proporção das classes:")
print(df['y_alfabetizado'].value_counts(normalize=True) * 100)

## 3. Remoção de Variáveis de Data Leakage
Como definido na extração, a coluna `leak_proficiencia` determina o target. Se mantida, o modelo terá 100% de acerto artificial. Também removeremos identificadores (`id_aluno`, `id_escola`, etc.) que não generalizam.

In [ ]:
# Definindo colunas a serem removidas para a modelagem
cols_to_drop = [
    'id_aluno', 'id_escola', 'id_municipio', 
    'y_alfabetizado_label', 'leak_proficiencia'
]
df_model = df.drop(columns=cols_to_drop)
print(f"Dataset reduzido de {df.shape[1]} para {df_model.shape[1]} colunas.")

## 4. Análise de Variáveis Categóricas
Vamos verificar a taxa de alfabetização por Rede (Estadual, Municipal, etc.) e por Região do Brasil.

In [ ]:
# Taxa de alfabetização por Rede
plt.figure(figsize=(8, 5))
sns.barplot(data=df_model, x='rede', y='y_alfabetizado', palette='Pastel1', errorbar=None)
plt.title('Taxa de Alfabetização por Rede de Ensino')
plt.ylabel('Proporção de Alfabetizados')
plt.show()

# Taxa de alfabetização por Região
plt.figure(figsize=(10, 5))
sns.barplot(data=df_model, x='nome_regiao', y='y_alfabetizado', palette='Pastel2', errorbar=None)
plt.title('Taxa de Alfabetização por Região')
plt.ylabel('Proporção de Alfabetizados')
plt.show()

## 5. Análise de Variáveis Numéricas (Porte)

In [ ]:
num_cols = ['qtd_alunos_escola', 'qtd_alunos_municipio', 'qtd_escolas_municipio']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(num_cols):
    sns.histplot(df_model[col], bins=30, ax=axes[i], kde=True)
    axes[i].set_title(f'Distribuição de {col}')
plt.tight_layout()
plt.show()

## 6. Correlação Numérica
Verificando a correlação de Spearman (já que as distribuições não são normais).

In [ ]:
plt.figure(figsize=(8, 6))
# Calculando correlação de Spearman para numéricas contínuas (inclusive o peso e target)
cols_corr = num_cols + ['w_peso_aluno', 'ref_meta_taxa_ano']
# Criando subset e tratando NaN temporariamente apenas para correlação
corr = df_model[['y_alfabetizado'] + cols_corr].astype(float).corr(method='spearman')
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação')
plt.show()

## 7. Conclusões da EDA
* Descrever os insights encontrados nos gráficos acima.
* A variável target é balanceada ou desbalanceada?
* Existe disparidade regional ou por rede?
* As variáveis de porte possuem forte correlação linear com o target, ou precisaremos do poder não-linear de modelos baseados em árvore?